# M1 — one AOD, a moving (astigmatic) tweezer

**What this notebook shows.** A single acousto-optic deflector — channel `Ay` of the
3D-AODL — driven first with a static tone and then with a minimum-jerk frequency sweep.
Three pieces of physics fall out, and each is checked against its closed-form prediction:

| # | Physics | Prediction (paper Table I / `docs/conventions.md`) |
|---|---------|------------------------------|
| 1 | **Deflection** — the tone position is the spot position | $Y = -\dfrac{\lambda F}{v}\,f(t_c)$ |
| 2 | **Chirp lensing** — a sweeping tone is a *cylindrical* lens | $Z_{y,\rm lab} = \dfrac{\lambda F^2}{v^2}\,\dot f(t_c)$, $\;\Delta F = Z_x - Z_y = -\dfrac{\lambda F^2}{v^2}\dot f$ |
| 3 | **Aperture transit** — the drive needs $\tau = D/v$ to fill the crystal | retarded time $t_c = t - \tau/2$; light builds up over a beam transit $2w_{\rm in}/v$ |

Point 2 is the M1 headline: **one AOD cannot move a tweezer without astigmatising it.**
The lens acts on the driven axis only, so the y focus walks out of the focal plane while
the x focus stays put, and the spot stretches. Cancelling that is what the *other three*
AODs are for (M2–M3).

Everything below runs through the package's ordinary front door — build a `WaveformSet`,
call `simulate`, read metrics or render frames. Physics reference: arXiv:2510.11451
(equations `S#` refer to its Supplement).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aodl import (
    ChannelWaveform,
    FrameGrid,
    PiecewisePoly,
    ToneTrack,
    WaveformSet,
    default_1030,
    ramps,
    render_movie,
    simulate,
)
from aodl.device.aod import aperture_window
from aodl.device.conventions import geometry
from aodl.field.focal import spot_params
from aodl.units import MHz, mm, um, us
from aodl.viz.style import composite, z_color

P = default_1030()          # paper hardware at lambda = 1030 nm (docs/PLAN.md 1.5)
optics, aod = P.optics, P.channels["Ay"]
tau = aod.transit_time      # acoustic transit across the aperture, D / v
OUT = Path("outputs")       # examples/outputs/ - gitignored

print(f"aperture D           = {aod.aperture / mm:6.2f} mm   at v = {aod.sound_speed:.0f} m/s")
print(f"transit time tau     = {tau / us:6.2f} us   (beam-centre retardation tau/2 = {tau / 2 / us:.2f} us)")
print(f"beam transit 2w_in/v = {2 * optics.w_in / aod.sound_speed / us:6.2f} us")
print(f"deflection scale     = {P.deflection_scale * MHz / um:6.2f} um per MHz   (lambda F / v)")
print(f"lens scale           = {P.lens_scale * (MHz / 1e-3) / um:6.3f} um per (MHz/ms)   (lambda F^2 / v^2)")
print(f"focal waist w0       = {optics.waist0 / um:6.3f} um,  Rayleigh range = {optics.rayleigh / um:.3f} um")

## 1. A static tone is a static tweezer

With one channel driven at detuning $f$, the pupil picks up a linear phase
$\theta_1 = s\,2\pi f/v$ (Eq. S6), and Eq. S11 maps a pupil tilt to an image position
$Y = \theta_1 F/k$. For `Ay` the sound travels toward $-y$ ($s = -1$), so

$$Y_{\rm spot} = -\frac{\lambda F}{v}\, f(t_c) = -10.3\ \mu\text{m} \times \frac{f}{\rm MHz}.$$

No chirp means no lens: the spot should sit exactly in the focal plane, round, at the
diffraction-limited waist $w_0 = \lambda F/(\pi w_{\rm in})$.

In [ ]:
detuning = 3.0 * MHz
static_wfs = WaveformSet(
    {"Ay": ChannelWaveform((ToneTrack(freq=PiecewisePoly.constant(detuning, 0.0, 4 * tau)),))},
    P,
)
static = simulate(static_wfs, [2 * tau])       # t = 2 tau: aperture fully filled
spot = static.metrics[0][0]

y_predicted = -P.deflection_scale * detuning
print(f"Y   measured {spot.y / um:+9.4f} um   predicted {y_predicted / um:+9.4f} um   (Table I)")
print(f"w_y measured {spot.wy / um:9.4f} um   waist0    {optics.waist0 / um:9.4f} um")
print(f"Zbar {spot.z_lab / um:+.2e} um,  Delta F {spot.delta_f / um:+.2e} um   -> no chirp, no lens")
assert abs(spot.y - y_predicted) < 1e-3 * optics.waist0
assert abs(spot.wy - optics.waist0) < 1e-6 * optics.waist0
assert abs(spot.wx - optics.waist0) < 1e-6 * optics.waist0

In [ ]:
half = 6 * optics.waist0
grid = FrameGrid(-half, half, 201, spot.y - half, spot.y + half, 201)
frame = static.frame(0, grid)

fig, ax = plt.subplots(figsize=(4.0, 4.0))
ax.imshow(
    composite([(frame, z_color(spot.z_lab, 1 * um))], frame.max()),
    extent=[c / um for c in grid.extent],
    origin="lower",
)
ax.axhline(y_predicted / um, color="w", lw=0.6, ls="--", alpha=0.6)
ax.set(xlabel="X [µm]", ylabel="Y [µm]", title=f"static tone, +{detuning / MHz:.0f} MHz on Ay")
ax.text(0.04, 0.04, "dashed: Table I prediction", transform=ax.transAxes, color="w", fontsize=8)
plt.show()

## 2. What is actually on the crystal

The device layer never guesses: it reads the **aperture window**, the literal RF waveform
sitting in the crystal at frame time $t$,

$$V(u) = \sum_n A_n(t_{\rm ret}(u))\cos\!\big(2\pi f_c\,t_{\rm ret}(u) + \varphi_n(t_{\rm ret}(u))\big),
\qquad t_{\rm ret}(u) = t - \frac{s\,u + D/2}{v},$$

zero wherever the sound has not arrived yet. The beam only sees the middle of it: the input
Gaussian has $w_{\rm in} = 2$ mm inside a $D = 7.5$ mm aperture, so **the beam is the aperture
stop that matters**, and the useful transit time is the beam's, not the crystal's.

Because the drive is chirping, the local frequency *varies across the aperture* — that
gradient, $\dot f \cdot 2w_{\rm in}/v$ across the beam, is exactly the quadratic pupil phase
that will act as a cylindrical lens in §4.

In [ ]:
sweep_span, sweep_time = 5.0 * MHz, 100.0 * us
freq = ramps.min_jerk(0.0, sweep_time, 0.0, sweep_span)       # Eq. S14
chirp_wfs = WaveformSet({"Ay": ChannelWaveform((ToneTrack(freq=freq),))}, P)
t_mid = 0.5 * sweep_time + 0.5 * tau                          # mid-sweep frame time
u, V = aperture_window(chirp_wfs.channels["Ay"], aod, geometry("Ay"), t_mid, n=30001)

f_local = aod.f_center + float(freq(t_mid - 0.5 * tau))                        # drive at the beam centre
across = float(freq.derivative()(t_mid - 0.5 * tau)) * 2 * optics.w_in / aod.sound_speed

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 3.4))
ax0.plot(u / mm, V, color="#3a7bd5", lw=0.3, alpha=0.5)
ax0.plot(u / mm, np.exp(-((u / optics.w_in) ** 2)), color="k", lw=1.8, label="input beam $e^{-u^2/w_{in}^2}$")
for edge in (-optics.w_in, optics.w_in):
    ax0.axvline(edge / mm, color="k", ls="--", lw=1.0)
ax0.set(xlabel="aperture coordinate u [mm]", ylabel="V(u)",
        title=f"acoustic column at t = {t_mid / us:.1f} µs (aperture full)")
ax0.text(0.02, 0.06, f"~{aod.aperture * f_local / aod.sound_speed:.0f} carrier periods across D:\nplotted as a band",
         transform=ax0.transAxes, fontsize=8)
ax0.legend(fontsize=8, loc="upper right")

zoom = np.abs(u) < 25 * um
ax1.plot(u[zoom] / um, V[zoom], color="#3a7bd5", lw=1.2)
ax1.set(xlabel="u [µm]", ylabel="V(u)", ylim=(-1.35, 1.35),
        title=f"zoom: {f_local / MHz:.2f} MHz at the beam centre, {across / MHz:+.2f} MHz across $2w_{{in}}$")
plt.tight_layout()
plt.show()

## 3. The fill transient

The drive starts at $t = 0$ with an empty crystal. Content exists only where
$s\,u \le v t - D/2$, so the leading wavefront sweeps across the aperture and the diffracted
power follows the **beam's** Gaussian integral:

$$\frac{P(t)}{P_\infty} = \tfrac{1}{2}\,\mathrm{erfc}\!\Big(\frac{\sqrt 2\,u_{\rm edge}}{w_{\rm in}}\Big),
\qquad u_{\rm edge} = s\,(v t - D/2).$$

Half power lands at $t = \tau/2$ — the moment the wavefront reaches the beam centre, which is
also why every device-layer quantity is evaluated at $t_c = t - \tau/2$. The rise takes about
a beam transit $2w_{\rm in}/v \approx 6.2\ \mu$s, and everything is over by $\tau = 11.54\ \mu$s
when the aperture is full.

In [ ]:
t_fill = np.linspace(0.0, 15.0 * us, 151)
fill = simulate(static_wfs, t_fill).spot_table()["power"]
fill = fill / fill[-1]

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(t_fill / us, fill, color="#3a7bd5", lw=2)
ax.axvline(tau / us, color="k", ls="--", lw=1)
ax.axhline(1.0, color="k", ls=":", lw=0.8)
beam_transit = 2 * optics.w_in / aod.sound_speed
ax.axvspan((tau - 2 * optics.w_in / aod.sound_speed) / 2 / us, (tau + beam_transit) / 2 / us,
           color="#f4a261", alpha=0.25)
ax.annotate(f"$\\tau$ = {tau / us:.2f} µs\naperture full", (tau / us, 0.45),
            xytext=(tau / us + 0.6, 0.35), fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.8))
ax.annotate(f"$2w_{{in}}/v$ = {beam_transit / us:.2f} µs", (tau / 2 / us, 0.92), ha="center", fontsize=9)
ax.plot([tau / 2 / us], [np.interp(tau / 2, t_fill, fill)], "o", color="k", ms=5)
ax.set(xlabel="t [µs]", ylabel="diffracted power / plateau", title="aperture fill transient",
       xlim=(0, 15), ylim=(-0.02, 1.08))
plt.tight_layout()
plt.show()

print(f"P(tau/2)/P_inf = {np.interp(tau / 2, t_fill, fill):.4f}   (erfc prediction: 0.5)")

## 4. A minimum-jerk sweep: the astigmatism appears

Now sweep the tone $0 \to 5$ MHz in 100 µs with the minimum-jerk profile of Eq. S14
(zero velocity *and* acceleration at both ends). Two predictions to check, both at the
**retarded** time $t_c = t - \tau/2$:

$$Y(t) = -\frac{\lambda F}{v} f(t_c), \qquad
Z_{y,\rm lab}(t) = \frac{\lambda F^2}{v^2}\,\dot f(t_c), \quad Z_{x,\rm lab} = 0 .$$

The frames run past the end of the programmed ramp, so the drive is extended with
`with_hold_until` first — outside its domain a `ToneTrack` clamp-holds and its phase would
stop advancing, and `simulate` refuses to render that rather than quietly faking it.

The last panel is the M1 result: while the tone moves, the beam is astigmatic, with
$\sigma_{\rm astig} = \Delta F / z_R$ reaching $\approx -2.8$ at peak chirp — the spot is nearly
three times wider along y than along x in the lab focal plane.

In [ ]:
frames = np.linspace(0.0, sweep_time + tau, 121)
sweep = simulate(chirp_wfs.with_hold_until(sweep_time + tau), frames)
tab = sweep.spot_table()

t_c = frames - 0.5 * tau                                    # beam-centre retarded time
y_pred = -P.deflection_scale * freq(t_c)                    # Table I
zy_pred = P.lens_scale * freq.derivative()(t_c)             # per-axis lab focus
zy = tab["z_lab"] - 0.5 * tab["delta_f"]                    # measured Z_y = Zbar - Delta F / 2
zx = tab["z_lab"] + 0.5 * tab["delta_f"]
# 1/e^2 radii in the *lab focal plane* z = 0, where the astigmatism is visible
radii = [np.ravel(spot_params(sweep.terms(i), optics, 0.0)[2:]) for i in range(len(frames))]
wx0, wy0 = np.array(radii).T

print(f"max |Y - prediction|      = {np.max(np.abs(tab['y'] - y_pred)) / optics.waist0:.2e} waists")
print(f"max |Z_y - prediction|    = {np.max(np.abs(zy - zy_pred)) / optics.rayleigh:.2e} Rayleigh ranges")
print(f"peak |sigma_astig|        = {np.max(np.abs(tab['sigma_astig'])):.3f}")
print(f"final Y                   = {tab['y'][-1] / um:.3f} um  (= -10.3 um/MHz x 5 MHz)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.4), sharex=True)
(ax_y, ax_z), (ax_w, ax_s) = axes

ax_y.plot(frames / us, y_pred / um, color="k", lw=3, alpha=0.25, label=r"$-\lambda F f(t_c)/v$")
ax_y.plot(frames / us, tab["y"] / um, color="#3a7bd5", lw=1.4, label="simulated")
ax_y.plot(frames / us, -P.deflection_scale * freq(frames) / um, color="#c1121f", lw=1.0, ls=":",
          label="un-retarded $f(t)$")
ax_y.set(ylabel="Y [µm]", title="deflection follows the retarded tone")
ax_y.legend(fontsize=8)

ax_z.plot(frames / us, zy_pred / um, color="k", lw=3, alpha=0.25, label=r"$\lambda F^2 \dot f(t_c)/v^2$")
ax_z.plot(frames / us, zy / um, color="#3a7bd5", lw=1.4, label=r"simulated $Z_{y,lab}$")
ax_z.plot(frames / us, zx / um, color="#8ac926", lw=1.4, label=r"simulated $Z_{x,lab}$")
ax_z.plot(frames / us, tab["z_lab"] / um, color="#f4a261", lw=1.2, ls="--", label=r"$\bar Z$ (tracked plane)")
ax_z.set(ylabel="Z lab [µm]", title="the chirp is a cylindrical lens (y only)")
ax_z.legend(fontsize=8)

ax_w.plot(frames / us, wx0 / um, color="#8ac926", lw=1.6, label="$w_x(z=0)$")
ax_w.plot(frames / us, wy0 / um, color="#3a7bd5", lw=1.6, label="$w_y(z=0)$")
ax_w.axhline(optics.waist0 / um, color="k", ls=":", lw=0.8)
ax_w.set(xlabel="t [µs]", ylabel="1/e² radius [µm]", title="the spot stretches while it moves")
ax_w.legend(fontsize=8)

ax_s.plot(frames / us, tab["sigma_astig"], color="#c1121f", lw=1.6)
ax_s.axhline(0.0, color="k", lw=0.8)
ax_s.set(xlabel="t [µs]", ylabel=r"$\sigma_{astig} = \Delta F / z_R$", title="astigmatism during the move")
plt.tight_layout()
plt.show()

## 5. The movie

`render_movie` composites one tinted layer per optical-frequency group: **hue is the spot's
own lab Z** (blue below the focal plane, white in it, red above) and **brightness is intensity**,
with a single global maximum across the movie so a dimming spot really looks dimmer. The side
panel is an XZ slice through the tweezer's row, and the strip underneath is the drive the beam
centre sees, cursor on now.

The view is `mode="fixed"` — the static lab focal plane, i.e. what a camera parked at $Z = 0$
would record — because that is where a cylindrical lens shows itself: the spot elongates along
y and fades as its y focus climbs away. The package default, `mode="tracked"`, follows
$\bar Z$ instead, and for a single AOD that plane sits half way between the two line foci, where
the spot is round but swollen (the circle of least confusion). Tracked mode comes into its own
from M3, when the astigmatism is cancelled and $\bar Z$ is a real focus again.

In [ ]:
from IPython.display import Video

movie = render_movie(sweep, OUT / "01_sweep.mp4", mode="fixed", fps=25, spectrogram_panel=True)
print(f"{movie}  ({movie.stat().st_size / 1e3:.0f} kB, {sweep.n_frames} frames)")
Video(str(movie), embed=True, html_attributes="controls loop")

## What M2 adds

One AOD buys one axis of motion and an unavoidable cylindrical lens. The next milestone adds
the second, crossed deflector (`Ax` + `Ay`, i.e. a conventional 2D-AOD):

* **diagonal moves** — chirping both axes equally makes the two cylindrical lenses add into a
  *spherical* one, so the spot leaves the focal plane along $\bar Z$ but stays round
  ($\Delta F = 0$): out-of-plane motion, no astigmatism;
* **arrays** — several tones per channel put a whole grid of tweezers in the focal plane at
  spacing $\lambda F \Delta f / v$, and the pupil being a *product* means every tone pair
  appears (the 16-ray picture of Fig. S6);
* **intermodulation** — expanding $e^{iCV}$ past first order gives IM3 ghosts at
  $f_i + f_j - f_k$, which Schroeder phases (Eq. S23) suppress.

Full 3D control — moving in x, y *and* z with $\Delta F = 0$ throughout — needs all four
channels and the Eq. S19 synthesis of M3.